In [7]:
import pandas as pd
import duckdb
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
import joblib
import os

In [ ]:
df = pd.read_csv('../data/secondary_sales.csv')

os.makedirs('../backend', exist_ok=True)
con = duckdb.connect('../backend/warehouse.db')
con.execute("CREATE TABLE IF NOT EXISTS fact_sales AS SELECT * FROM df")
con.close()

X = df[['bedrooms', 'area_sqft', 'year_built', 'parking_spaces', 'to_burj_khalifa_km', 'property_category', 'furnishing']]
y_reg = df['price_usd']
y_clf = df['is_freehold'].astype(int)

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, stratify=y_clf, random_state=42
)

numeric_features = ['bedrooms', 'area_sqft', 'year_built', 'parking_spaces', 'to_burj_khalifa_km']
categorical_features = ['property_category', 'furnishing']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])

from sklearn.linear_model import Ridge, LogisticRegression

reg_pipeline = Pipeline([('prep', preprocessor), ('modelo', Ridge())])
reg_pipeline.fit(X_train, y_reg_train)

clf_pipeline = Pipeline([('prep', preprocessor), ('modelo', LogisticRegression(class_weight='balanced'))])
clf_pipeline.fit(X_train, y_clf_train)

joblib.dump(reg_pipeline, '../backend/modelo_regresion.pkl')
joblib.dump(clf_pipeline, '../backend/modelo_clasificacion.pkl')
print("✅ Modelos exportados con éxito al backend.")

✅ Modelos exportados con éxito al backend.
